# Career Analytics ETL and Data Model

In [1]:
#Extract

import json
import pandas as pd

SOURCE = "full-professionals-json.json"

with open(SOURCE) as f:
    raw = json.load(f)

docs = raw["professionals"]
print("documents:", len(docs))
print("top level keys:" , list(docs[0].keys()))


pd.DataFrame([
    {"professional_id": d["professional_id"],
     **{k: len(d.get(k) or []) for k in ["jobs", "skills", "certifications", "education"]}}
    for d in docs
])

ImportError: DLL load failed while importing np_datetime: An Application Control policy has blocked this file.

In [ ]:
#Flatten

PROFESSIONAL_COLS = ["professional_id", "years_experience", "current_industry",
                     "current_role", "education_level"]

CHILD_COLS = {
    "jobs": ["job_id", "company", "industry", "role", "start_date", "end_date", "salary_band"],
    "skills": ["skill_id", "skill_name", "proficiency_level", "years_experience"],
    "certifications": ["certification_id", "certification_name", "issuing_organization",
                       "date_earned", "expiration_date"],
    "education": ["education_id", "degree", "institution", "field_of_study", "graduation_date"],
}

# Parent table: one row per professional, plain fields only
tables = {
    "professionals": pd.DataFrame(
        [{c: d.get(c) for c in PROFESSIONAL_COLS} for d in docs],
        columns=PROFESSIONAL_COLS,
    )
}

# Child tables: one row per item in each nested list, tagged with the owner's professional_id
for child, cols in CHILD_COLS.items():
    rows = [
        {"professional_id": d.get("professional_id"), **{c: item.get(c) for c in cols}}
        for d in docs
        for item in (d.get(child) or [])
    ]
    tables[child] = pd.DataFrame(rows, columns=["professional_id", *cols])

for name, df in tables.items():
    print(f"{name:15s} {len(df):3d} rows")

tables["skills"].head()

professionals    10 rows
jobs             32 rows
skills           34 rows
certifications   19 rows
education        16 rows


,professional_id,skill_id,skill_name,proficiency_level,years_experience
0,P001,S001,Python,Advanced,8
1,P001,S002,SQL,Expert,10
2,P001,S003,Machine Learning,Advanced,6
3,P001,S004,Data Visualization,Intermediate,7
4,P001,S088,TensorFlow,Advanced,5


In [ ]:
#Transform

DATE_COLS = {
    "jobs": ["start_date", "end_date"],
    "certifications": ["date_earned", "expiration_date"],
    "education": ["graduation_date"],
}
INT_COLS = {
    "professionals": ["years_experience"],
    "jobs": ["salary_band"],
    "skills": ["years_experience"],
}

# CURRENT_ROLE is a reserved word in SQL, so rename it where the data enters
tables["professionals"] = tables["professionals"].rename(columns={"current_role": "current_role_title"})

for name, df in tables.items():
    date_cols = DATE_COLS.get(name, [])
    int_cols = INT_COLS.get(name, [])

    for c in date_cols:
        if f"{c}_raw" not in df.columns:
            df[f"{c}_raw"] = df[c]                      # keep the original text
        df[c] = pd.to_datetime(df[c], errors="coerce")  # invalid -> empty

    for c in int_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    for c in df.columns:
        if c.endswith("_raw") or c in date_cols or c in int_cols:
            continue
        df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

print(tables["jobs"].dtypes)
print()
print("jobs with no end_date (still current):", tables["jobs"]["end_date"].isna().sum())
tables["jobs"].head()

professional_id               str
job_id                        str
company                       str
industry                      str
role                          str
start_date         datetime64[us]
end_date           datetime64[us]
salary_band                 Int64
start_date_raw                str
end_date_raw                  str
dtype: object

jobs with no end_date (still current): 9


,professional_id,job_id,company,industry,role,start_date,end_date,salary_band,start_date_raw,end_date_raw
0,P001,J001,TechCorp,Technology,Junior Analyst,2011-06-15,2013-08-30,2,2011-06-15,2013-08-30
1,P001,J002,DataWorks,Technology,Data Analyst,2013-09-15,2016-05-20,3,2013-09-15,2016-05-20
2,P001,J003,Analytix,Technology,Senior Data Analyst,2016-06-01,2019-04-15,4,2016-06-01,2019-04-15
3,P001,J004,AIVentures,Technology,Data Scientist,2019-05-01,2023-02-28,5,2019-05-01,2023-02-28
4,P002,J005,MarketBoost,Marketing,Marketing Assistant,2015-03-10,2017-07-20,2,2015-03-10,2017-07-20


In [ ]:
jobs = tables["jobs"]

print("latest job start:", jobs["start_date"].max().date())
print("latest job end:  ", jobs["end_date"].max().date())

# Reference date for open jobs (no end_date): measured up to today
AS_OF_DATE = pd.Timestamp.today().normalize()
print("reference date:  ", AS_OF_DATE.date())

latest job start: 2022-04-15
latest job end:   2023-02-28
reference date:   2026-10-03


In [ ]:
#Data Quality checks

ID_COLS = {"professionals": "professional_id", "jobs": "job_id", "skills": "skill_id",
           "certifications": "certification_id", "education": "education_id"}

# Order each person's jobs and work out tenure and the previous job's end
jobs = tables["jobs"].sort_values(["professional_id", "start_date"]).reset_index(drop=True)
jobs["effective_end"] = jobs["end_date"].fillna(AS_OF_DATE)
jobs["tenure_days"] = (jobs["effective_end"] - jobs["start_date"]).dt.days
jobs["prev_end"] = jobs.groupby("professional_id")["effective_end"].shift()
tables["jobs"] = jobs

REMOVE = "Removed from the load"
REVIEW = "Kept, needs review"
findings = []

def check(df, is_bad, what_is_wrong, what_we_do, id_col, show):
    for _, row in df[is_bad.fillna(False)].iterrows():
        findings.append({"what is wrong": what_is_wrong, "professional": row["professional_id"],
                         "record": row[id_col], "value": row[show], "what we do": what_we_do})

prof, skills = tables["professionals"], tables["skills"]

# IDs present and unique
for name, df in tables.items():
    id_col = ID_COLS[name]
    check(df, df[id_col].isna(), f"A row in {name} has no ID", REMOVE, id_col, id_col)
    check(df, df[id_col].duplicated(keep=False), f"The same ID appears twice in {name}", REMOVE, id_col, id_col)

# Dates that were present but could not be read
for name, cols in DATE_COLS.items():
    for c in cols:
        df = tables[name]
        check(df, df[f"{c}_raw"].notna() & df[c].isna(), f"{c} is not a valid date", REMOVE, ID_COLS[name], f"{c}_raw")

# Date logic within a person's job history
check(jobs, jobs["end_date"] < jobs["start_date"], "Job ends before it starts", REMOVE, "job_id", "role")
check(jobs, jobs["start_date"] < jobs["prev_end"], "Job starts before the previous one ended", REVIEW, "job_id", "role")

# Outliers
check(jobs, jobs["salary_band"].isna() | ~jobs["salary_band"].between(1, 10),
      "Salary band is missing or outside 1 to 10", REMOVE, "job_id", "salary_band")
check(jobs, (jobs["end_date"].notna() & (jobs["tenure_days"] < 90)) | (jobs["tenure_days"] > 15 * 365),
      "Job lasted under 90 days or over 15 years", REVIEW, "job_id", "tenure_days")

# Summary says there is a current role, but no job is open
open_job_owners = jobs.loc[jobs["end_date"].isna(), "professional_id"].unique()
check(prof, prof["current_role_title"].notna() & ~prof["professional_id"].isin(open_job_owners),
      "Has a current role but every job has ended", REVIEW, "professional_id", "current_role_title")

# Same skill name stored under different IDs
check(skills, skills.groupby("skill_name")["skill_id"].transform("nunique") > 1,
      "Same skill is stored under different IDs", REVIEW, "skill_id", "skill_name")

findings_df = pd.DataFrame(findings, columns=["what is wrong", "professional", "record", "value", "what we do"])
print(f"Checked {sum(len(df) for df in tables.values())} rows across {len(tables)} tables.")
print(f"{(findings_df['what we do'] == REMOVE).sum()} rows removed, {(findings_df['what we do'] == REVIEW).sum()} rows kept for review.")
findings_df

Checked 111 rows across 5 tables.
0 rows removed, 5 rows kept for review.


,what is wrong,professional,record,value,what we do
0,Has a current role but every job has ended,P001,P001,Data Scientist,"Kept, needs review"
1,Same skill is stored under different IDs,P001,S001,Python,"Kept, needs review"
2,Same skill is stored under different IDs,P003,S089,Python,"Kept, needs review"
3,Same skill is stored under different IDs,P004,S011,Financial Modeling,"Kept, needs review"
4,Same skill is stored under different IDs,P009,S091,Financial Modeling,"Kept, needs review"


In [ ]:
#Dimension tables

jobs, skills, certs, prof = (tables[t] for t in ["jobs", "skills", "certifications", "professionals"])

def make_dim(values, name):
    unique = sorted(values.dropna().unique())
    return pd.DataFrame({f"{name}_key": range(1, len(unique) + 1), f"{name}_name": unique})

dim_company = make_dim(jobs["company"], "company")
dim_industry = make_dim(jobs["industry"], "industry")
dim_role = make_dim(jobs["role"], "role")
dim_skill = make_dim(skills["skill_name"], "skill")

dim_professional = prof.copy()
dim_professional.insert(0, "professional_key", range(1, len(prof) + 1))

# number of certifications each person holds
cert_counts = certs.groupby("professional_id").size()
dim_professional["certification_count"] = (
    dim_professional["professional_id"].map(cert_counts).fillna(0).astype(int)
)

for name, df in [("dim_professional", dim_professional), ("dim_company", dim_company),
                 ("dim_industry", dim_industry), ("dim_role", dim_role), ("dim_skill", dim_skill)]:
    print(f"{name:18s} {len(df):3d} rows")

dim_company

NameError: name 'tables' is not defined

In [ ]:
# fact tables

def key_for(values, dim, name):
    return values.map(dim.set_index(f"{name}_name")[f"{name}_key"])

professional_key = dim_professional.set_index("professional_id")["professional_key"]

# Each job compared with the same person's previous job
previous = jobs.groupby("professional_id")[["salary_band", "tenure_days"]].shift()

fact_job = pd.DataFrame({
    "job_id": jobs["job_id"],
    "professional_key": jobs["professional_id"].map(professional_key),
    "company_key": key_for(jobs["company"], dim_company, "company"),
    "industry_key": key_for(jobs["industry"], dim_industry, "industry"),
    "role_key": key_for(jobs["role"], dim_role, "role"),
    "start_date": jobs["start_date"],
    "end_date": jobs["end_date"],
    "salary_band": jobs["salary_band"],
    "tenure_days": jobs["tenure_days"],
    "band_change": jobs["salary_band"] - previous["salary_band"],
    "days_in_previous_job": previous["tenure_days"],
})

fact_professional_skill = pd.DataFrame({
    "professional_key": skills["professional_id"].map(professional_key),
    "skill_key": key_for(skills["skill_name"], dim_skill, "skill"),
    "proficiency_level": skills["proficiency_level"],
    "years_experience": skills["years_experience"],
})

fact_certification = pd.DataFrame({
    "professional_key": certs["professional_id"].map(professional_key),
    "certification_name": certs["certification_name"],
    "date_earned": certs["date_earned"],
})

print(len(fact_job), "jobs,", len(fact_professional_skill), "skills,", len(fact_certification), "certifications")
fact_job.head()

32 jobs, 34 skills, 19 certifications


,job_id,professional_key,company_key,industry_key,role_key,start_date,end_date,salary_band,tenure_days,band_change,days_in_previous_job
0,J001,1,30,5,15,2011-06-15,2013-08-30,2,807,<NA>,NaN
1,J002,1,12,5,6,2013-09-15,2016-05-20,3,978,1,807.0
2,J003,1,2,5,25,2016-06-01,2019-04-15,4,1048,1,978.0
3,J004,1,1,5,7,2019-05-01,2023-02-28,5,1399,1,1048.0
4,J005,2,22,3,17,2015-03-10,2017-07-20,2,863,<NA>,NaN


In [ ]:
# Load into duckdb

import duckdb

con = duckdb.connect("career_analytics.duckdb")

model = {
    "dim_professional": dim_professional,
    "dim_company": dim_company,
    "dim_industry": dim_industry,
    "dim_role": dim_role,
    "dim_skill": dim_skill,
    "fact_job": fact_job,
    "fact_professional_skill": fact_professional_skill,
    "fact_certification": fact_certification,
}

for name, df in model.items():
    con.register("incoming", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM incoming")
    con.unregister("incoming")
    rows = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:25s} {rows:3d} rows")

dim_professional           10 rows
dim_company                32 rows
dim_industry                5 rows
dim_role                   31 rows
dim_skill                  32 rows
fact_job                   32 rows
fact_professional_skill    34 rows
fact_certification         19 rows


# Approach

The source file has one document per professional, with jobs, skills, certifications and education nested inside it.
1. Read the JSON and checked its structure.
2. Flattened each nested list into its own table, linked by professional_id.
3. Converted dates and numbers to proper types. 
4. Ran data quality checks on IDs, dates, outliers and consistency.
5. Built a star schema: five dimensions (professional, company, industry, role, skill) and three facts (job, professional skill, certification).
6. Loaded the tables into DuckDB with a full refresh.

# Assumptions

1. Jobs with no end date are measured up to today. The data looks like it is from early 2023, so tenure for current jobs is overstated. In production this would be the pipeline run date.
2. Jobs are ordered by start date, and each is compared with the one before it.
3. Companies, roles and skills are matched on exact name.
4. Outlier limits are fixed: salary band 1 to 10, tenure 90 days to 15 years.


# Metrics are calculated below

In [ ]:

#Salary bands gained per year, by industry

con.execute("""
    WITH person AS (
        SELECT f.professional_key, i.industry_name,
               MAX(f.salary_band) - MIN(f.salary_band) AS bands_gained,
               SUM(f.tenure_days) / 365.25 AS years_worked
        FROM fact_job f
        JOIN dim_industry i ON i.industry_key = f.industry_key
        GROUP BY f.professional_key, i.industry_name
    )
    SELECT industry_name,
           COUNT(*) AS professionals,
           ROUND(AVG(bands_gained / years_worked), 2) AS bands_gained_per_year
    FROM person
    GROUP BY industry_name
    ORDER BY bands_gained_per_year DESC
""").df()

,industry_name,professionals,bands_gained_per_year
0,Technology,4,0.24
1,Retail,1,0.17
2,Marketing,2,0.17
3,Healthcare,1,0.15
4,Finance,2,0.13


In [ ]:
#Time to promotion, by industry

con.execute("""
    SELECT i.industry_name,
           COUNT(*) AS promotions,
           ROUND(AVG(f.days_in_previous_job) / 30.44, 1) AS avg_months_to_promotion
    FROM fact_job f
    JOIN dim_industry i ON i.industry_key = f.industry_key
    WHERE f.band_change > 0
    GROUP BY i.industry_name
    ORDER BY avg_months_to_promotion
""").df()

,industry_name,promotions,avg_months_to_promotion
0,Marketing,3,26.9
1,Healthcare,2,28.2
2,Technology,11,30.8
3,Retail,2,35.4
4,Finance,4,41.2


In [ ]:
# Growth by education level and certifications

con.execute("""
    WITH person AS (
        SELECT professional_key,
               MAX(salary_band) - MIN(salary_band) AS bands_gained,
               SUM(tenure_days) / 365.25 AS years_worked
        FROM fact_job
        GROUP BY professional_key
    )
    SELECT p.education_level,
           COUNT(*) AS professionals,
           ROUND(AVG(p.certification_count), 1) AS avg_certifications,
           ROUND(AVG(person.bands_gained / person.years_worked), 2) AS bands_gained_per_year
    FROM person
    JOIN dim_professional p ON p.professional_key = person.professional_key
    GROUP BY p.education_level
    ORDER BY bands_gained_per_year DESC
""").df()

,education_level,professionals,avg_certifications,bands_gained_per_year
0,Masters,3,3.0,0.22
1,Bachelors,5,1.2,0.18
2,PhD,1,1.0,0.15
3,MBA,1,3.0,0.14
